# Vignette 1: Zero-shot generation across backends

Synderm 0.2 wraps three open-weight image generators behind one interface:

| backend | model | text-to-image | reference-image edit | hard-mask inpaint |
|---|---|---|---|---|
| `qwen-image-2.1` | Qwen-Image-2.1 (Alibaba, Sept 2026, 7B) | yes | yes | no |
| `flux-3-klein` | FLUX 3 Image Klein (Black Forest Labs, Oct 2026, 5.6B) | yes | yes | no |
| `sdxl` | Stable Diffusion XL 1.0 (Stability AI, 2023) | yes | no | yes |

This notebook generates one image per disease label with each backend **without any
fine-tuning**, next to a real example from Fitzpatrick 17k. It shows how to load a backend on a
32 GB card, what the default prompts look like, and how the three models differ before any
adaptation. Later vignettes fine-tune them.

All three models are loaded one after another; each one is unloaded before the next.

## Parameters

`SMALL=True` keeps the run short (three labels, one image each). Set `USE_PROMPT_UPSAMPLER=False`
to skip the FLUX 3 prompt upsampler (a separate 9B model, ~19 GB download) and use the
clinical caption template instead.

In [ ]:
SMALL = True
RESOLUTION = 768
SEED = 0
USE_PROMPT_UPSAMPLER = True
BACKENDS = ["sdxl", "qwen-image-2.1", "flux-3-klein"]

In [ ]:
import os, json, time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image

os.environ.setdefault("HF_HOME", "/workspace/.cache/huggingface")
WORK = Path(os.environ.get("SYNDERM_WORK_DIR", "/workspace/synderm"))
WORK.mkdir(parents=True, exist_ok=True)
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

def show_grid(rows, row_titles=None, col_titles=None, size=2.6, suptitle=None):
    """rows: list of lists of PIL images (one list per row)."""
    n_rows, n_cols = len(rows), max(len(r) for r in rows)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(size * n_cols, size * n_rows), squeeze=False)
    for r, row in enumerate(rows):
        for c in range(n_cols):
            ax = axes[r][c]; ax.axis("off")
            if c < len(row) and row[c] is not None:
                ax.imshow(row[c])
            if r == 0 and col_titles and c < len(col_titles):
                ax.set_title(col_titles[c], fontsize=10)
        if row_titles:
            axes[r][0].text(-0.08, 0.5, row_titles[r], transform=axes[r][0].transAxes, rotation=90,
                            va="center", ha="right", fontsize=10)
    if suptitle:
        fig.suptitle(suptitle)
    plt.tight_layout(); plt.show()

## 1. Real reference images

`load_sample` fetches a fixed six-label sample of Fitzpatrick 17k from Atlas Dermatologico
(64 train + 32 validation images per label), verifies every file against its md5 hash and
caches it under `/workspace/data`. It raises if the source is unreachable; there is no
fallback, so an unavailable dataset is noticed rather than hidden.

In [ ]:
from synderm.data import load_sample
real = load_sample(split="train")
labels = real.labels if not SMALL else real.labels[:3]
print(len(real), "real training images;", "labels:", labels)
examples = {}
for i in range(len(real)):
    lab = real.df.iloc[i]["label"]
    if lab in labels and lab not in examples:
        examples[lab] = real[i]["image"]
show_grid([[examples[l] for l in labels]], col_titles=labels, suptitle="Real (Fitzpatrick 17k / Atlas Dermatologico)")

## 2. Prompts

Each backend has a default caption template. SDXL and Qwen use the paper's prompt
(`"An image of {condition}, a skin disease"`). FLUX 3 Klein expects full scene captions, so its
default is a longer clinical template, and its official prompt upsampler can expand prompts
further (`upsample_prompts`).

In [ ]:
from synderm.backends import get_backend, list_backends
for info in list_backends():
    print(f"{info.name:16s} {info.model_id:48s} edit={info.supports_edit!s:5s} inpaint={info.supports_inpaint!s:5s} license={info.license}")
print()
for name in BACKENDS:
    print(name, "->", get_backend(name).caption("psoriasis"))

## 3. Generate one image per label with each backend

`text_to_image` takes a prompt (or list), a seed and an output size, and returns PIL images.
Loading a model from the network filesystem takes a couple of minutes the first time.

In [ ]:
outputs = {}
timings = {}
for name in BACKENDS:
    gen = get_backend(name)
    t = time.time(); gen.load(); load_s = time.time() - t
    prompts = [gen.caption(l) for l in labels]
    if name == "flux-3-klein" and USE_PROMPT_UPSAMPLER:
        prompts = gen.upsample_prompts(prompts)
        print("Upsampled caption for", labels[0], ":\n", prompts[0][:600], "...\n")
    torch.cuda.reset_peak_memory_stats(); t = time.time()
    imgs = gen.text_to_image(prompts, num_images_per_prompt=1, seed=SEED, height=RESOLUTION, width=RESOLUTION)
    gen_s = (time.time() - t) / len(imgs)
    outputs[name] = imgs
    timings[name] = {"load_s": round(load_s), "s_per_image": round(gen_s, 1), "peak_vram_gb": round(torch.cuda.max_memory_allocated() / 1e9, 1)}
    gen.unload()
    print(name, timings[name])
pd.DataFrame(timings).T

In [ ]:
rows = [[examples[l] for l in labels]] + [outputs[n] for n in BACKENDS]
show_grid(rows, row_titles=["real"] + BACKENDS, col_titles=labels, suptitle=f"Zero-shot, {RESOLUTION}px, seed {SEED}")

## What to look for

- Zero-shot, none of the models has seen clinical photographs labelled with these diagnoses in a
  controlled way; they draw on whatever dermatology imagery was in their pretraining data.
- SDXL (2023) tends to produce generic "skin texture" images. The 2026 models produce more
  photographic, better composed images, but label fidelity is still an open question that the
  quality vignette (06) measures.
- The paper found physician-rated photorealism correlates with classifier gains; vignette 02
  fine-tunes a LoRA so the generators learn the dataset's appearance.

In [ ]:
out_dir = WORK / "vignettes" / "01_zero_shot"
out_dir.mkdir(parents=True, exist_ok=True)
for n, imgs in outputs.items():
    for l, im in zip(labels, imgs):
        im.save(out_dir / f"{n}-{l}.png")
print("saved to", out_dir)